In [1]:
# Soil Erosion Prediction - Full Training Pipeline

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
import joblib

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("AWESOME_V1.csv", encoding='ISO-8859-1')

# Interpolate missing values
df = df.interpolate(method='nearest')

# Drop rows missing important columns
required_cols = ['MAT','MAP','Elevation','Latitude','Longitude','Slope','Soil_sand','Soil_silt','Soil_clay','Soil_SOC']
df = df.dropna(subset=required_cols)

# -----------------------------
# Feature Selection
# -----------------------------
X = df[['MAT','MAP','Elevation','Latitude','Longitude','Slope','Soil_sand','Soil_silt','Soil_clay']]
y = df['Soil_SOC']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -----------------------------
# Preprocessing Pipeline
# -----------------------------
numeric_features = X.columns.tolist()
numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

preprocess = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features)
])

# -----------------------------
# Models to Compare
# -----------------------------
models = {
    "RandomForest": RandomForestRegressor(),
    "GradientBoosting": GradientBoostingRegressor(),
    "ElasticNet": ElasticNet(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR()
}

param_grid = {
    "RandomForest": {
        'model__n_estimators': [100, 200],
        'model__max_depth': [10, 20],
        'model__min_samples_split': [2, 5],
        'model__min_samples_leaf': [1, 2]
    },
    "GradientBoosting": {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1, 0.2],
        'model__max_depth': [3, 5]
    },
    "ElasticNet": {
        'model__alpha': [0.01, 0.1, 1.0],
        'model__l1_ratio': [0.2, 0.5, 0.8]
    },
    "KNN": {
        'model__n_neighbors': [3, 5, 7]
    },
    "SVR": {
        'model__C': [1, 10],
        'model__kernel': ['rbf']
    }
}

# -----------------------------
# Train & Select Best Model
# -----------------------------
best_model = None
best_score = -999
best_name = ""

for name, model in models.items():
    print(f"\nTraining {name}...")

    pipeline = Pipeline([
        ('preprocess', preprocess),
        ('model', model)
    ])

    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid[name],
        n_iter=8,
        cv=5,
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train, y_train)
    preds = search.predict(X_test)

    r2 = r2_score(y_test, preds)
    print(f"{name} R² Score = {r2}")

    if r2 > best_score:
        best_score = r2
        best_model = search.best_estimator_
        best_name = name

# -----------------------------
# Final Output
# -----------------------------
print("\n==============================")
print(" BEST MODEL SELECTED ")
print("==============================")
print("Model:", best_name)
print("Best R² Score:", best_score)

# -----------------------------
# Save Pickle File
# -----------------------------
joblib.dump(best_model, "soil_erosion_best_model.pkl")
print("\nPickle file saved as: soil_erosion_best_model.pkl")


C:\Users\Hp\AppData\Local\Temp\ipykernel_17012\771173118.py:22: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.interpolate(method='nearest')



Training RandomForest...
RandomForest R² Score = 0.8986118521169703

Training GradientBoosting...
GradientBoosting R² Score = 0.9300251330147586

Training ElasticNet...
ElasticNet R² Score = 0.03258616289949179

Training KNN...


D:\Software\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=8. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


KNN R² Score = 0.8194282767732753

Training SVR...


D:\Software\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=8. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


SVR R² Score = 0.6222254313178701

 BEST MODEL SELECTED 
Model: GradientBoosting
Best R² Score: 0.9300251330147586

Pickle file saved as: soil_erosion_best_model.pkl
